In [2]:
import xarray as xr
import pandas as pd
import os
import glob
from pathlib import Path
import re
import datetime
import numpy as np
import matplotlib.pyplot as plt

# import importlib
# import get_data
# # Make changes to my_module.py
# importlib.reload(get_data)
from get_data import get_alldata, weekmean, calcroll_anomERA5

## Get CESM2 S2S Hindcast Data

In [9]:
ddir = '/glade/campaign/cesm/development/cross-wg/S2S/CESM2/S2SHINDCASTS/p1/tas_2m/'
save_ddir = '/glade/derecho/scratch/kjmayer/DATA/CESM2-S2S/'
combined_da = get_alldata(ddir)

for week in ['init','week12','week34','week56']:
    
    save_finame = 'tas_2m_anom_cesm2cam6v2_04jan1999-28dec2020_00z_'+week+'mean_m00-10mean.nc'
    da_weekavg = weekmean(data=combined_da,
                          week=week,
                          ddir=save_ddir,
                          finame=save_finame,
                          save=True
                         )

File '/glade/derecho/scratch/kjmayer/DATA/CESM2-S2S/tas_2m_anom_cesm2cam6v2_04jan1999-28dec2020_00z_initmean_m00-10mean.nc' exists.
member mean calculated
anomalies calculated
lead week mean calculated
mean saved
File '/glade/derecho/scratch/kjmayer/DATA/CESM2-S2S/tas_2m_anom_cesm2cam6v2_04jan1999-28dec2020_00z_week34mean_m00-10mean.nc' exists.
member mean calculated
anomalies calculated
lead week mean calculated
mean saved


## ERA5 Preprocess:

In [6]:
weekslice = {'init': 0,
             'week12': [0, 13],
             'week34': [14, 27],
             'week56': [28, 41],
             }

ddir_obs = '/glade/campaign/cesm/development/cross-wg/S2S/sglanvil/data/'
finame_obs = 'tas_2m_ERA5_19990101-20211231.nc'
obs = xr.open_dataset(ddir_obs+finame_obs)['tas_2m']
obs['time'] = pd.date_range('1999-01-01','2021-12-31')
obs_anom = calcroll_anomERA5(obs)

In [10]:
ddir_mod = '/glade/derecho/scratch/kjmayer/DATA/CESM2-S2S/'

for week in ['week12','week34','week56']:
    finame_mod = 'tas_2m_anom_cesm2cam6v2_04jan1999-28dec2020_00z_'+week+'mean_m00-10mean.nc'
    model_weekmean_anom = xr.open_dataset(ddir_mod+finame_mod)['tas_2m']
    init = model_weekmean_anom['init']

    time_slice = weekslice.get(week)
    weekstrt = init + pd.Timedelta(days=time_slice[0])
    weekend = init + pd.Timedelta(days=time_slice[-1])

    i_init = 0
    weekcombined = []
    for strt, end in zip(weekstrt, weekend):
        weekmean_temp = obs_anom.sel(time=slice(strt, end)).mean('time')
        weekmean_temp = weekmean_temp.assign_coords(init=(init[i_init]))
        weekcombined.append(weekmean_temp)
        i_init += 1
    obs_weekmean_anom = xr.concat(weekcombined, dim="init")

    finame_save = 'tas_2m_anom_ERA5_04jan1999-28dec2020_'+week+'mean.nc'
    obs_weekmean_anom.to_netcdf(ddir_mod+finame_save)